In [33]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import subprocess
import sys
from pathlib import Path

package_dir = Path.cwd().resolve()
if package_dir.name == 'examples':
    package_dir = package_dir.parent
else:
    repo_candidate = Path('packages/visualization/calibrated-explanations-visualization-plotly').resolve()
    if repo_candidate.exists():
        package_dir = repo_candidate

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(package_dir)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly>=5.18'])

0

# Local Ensured Triangular Plotly

This notebook demonstrates `plotly.local.ensured_triangular`, a Plotly version of CE's existing ensured or triangular local alternative plot.

Plot semantics stay aligned with the native view:

- x-axis = probability in probabilistic mode, prediction in regression mode
- y-axis = uncertainty
- red marker = original prediction
- blue markers = alternative or rule points
- arrows = predictive movement from the original point to shown rule alternatives
- hover = rule conditions plus calibrated interval metadata

The arrows do not imply causal actionability. They visualize predictive movement under alternative rule conditions.

`filter_top` keeps dense ensured plots readable by limiting how many rule points and arrows are shown.

In [35]:
import numpy as np
from calibrated_explanations import WrapCalibratedExplainer
from ce_visualization_plotly.plugin import register_plotly_visualization_components
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

register_plotly_visualization_components()
np.set_printoptions(precision=3, suppress=True)

## Data Split

Generate a binary classification dataset and split it into proper training, calibration, and query data using a 60/20/20 split. Calibration data stays separate from the model fitting data.

In [36]:
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    random_state=7,
)

x_proper, x_holdout, y_proper, y_holdout = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=7,
    stratify=y,
)
x_cal, X_query, y_cal, y_query = train_test_split(
    x_holdout,
    y_holdout,
    test_size=0.50,
    random_state=7,
    stratify=y_holdout,
)

x_proper.shape, x_cal.shape, X_query.shape

((300, 8), (100, 8), (100, 8))

## Fit and Calibrate

Use `WrapCalibratedExplainer` rather than `CalibratedExplainer` directly. The assertions make the fitted and calibrated states explicit before explanations are requested.

In [37]:
model = LogisticRegression(random_state=7, solver='liblinear')
explainer = WrapCalibratedExplainer(model)
explainer.fit(x_proper, y_proper)
assert explainer.fitted is True

explainer.calibrate(x_cal, y_cal)
assert explainer.calibrated is True

## Generate Alternative Explanations

The ensured or triangular view is defined over alternative explanations, so this notebook uses `explore_alternatives(...)`. For the current CE collection-level custom plot routing, omit `instance_index` here so the Plotly plugin path stays active and renders the first explanation in the collection.

In [38]:
alternatives = explainer.explore_alternatives(X_query[:3])
len(alternatives.explanations)

3

## Interactive Plot

Hover now shows a compact summary: rule, prediction, uncertainty, and interval.

In [ ]:
plot_result = alternatives.plot(
    style='plotly.local.ensured_triangular',
    show=True,
    filter_top=20,
)

## HTML Export

The current collection-level API uses `filename=`. That value is forwarded into the plugin renderer, which exports the interactive Plotly figure as HTML.

In [ ]:
export_result = alternatives.plot(
    style='plotly.local.ensured_triangular',
    show=False,
    filename='ensured_triangular.html',
    filter_top=20,
)
export_result.saved_paths

('ensured_triangular.html',)